In [1]:
!pip install skl2onnx onnxmltools onnxruntime lz4 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.0/304.0 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 63.9 MB/s eta 0:00:00


In [5]:
import pandas as pd
import numpy as np
import time
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, f1_score, recall_score, accuracy_score
from xgboost import XGBClassifier

# ── Load preprocessed data ──────────────────────────────────────────────────
processed_df = pd.read_csv(
    '/content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smotenc(hybrid70-30)/preprocessed_dataset_with_shadow.csv'
)

X        = processed_df.drop(columns=['Binary_Label', 'Superclass', 'Fine_Label'])
y_binary = processed_df['Binary_Label']
s        = processed_df['Superclass']

X_train, X_test, y_train, y_test, s_train, s_test = train_test_split(
    X, y_binary, s,
    test_size=0.2,
    stratify=processed_df['Fine_Label'],
    random_state=42
)

# ── Train XGBoost (same params as original) ─────────────────────────────────
model = XGBClassifier(n_estimators=50, max_depth=6, verbosity=0)
print("🚀 Training XGBoost...")
model.fit(X_train, y_train)

# ── Evaluate ─────────────────────────────────────────────────────────────────
start_inf = time.time()
y_pred    = model.predict(X_test)
end_inf   = time.time()

total_inference_time_s = end_inf - start_inf
avg_latency_ms         = (total_inference_time_s / len(X_test)) * 1000
throughput             = len(X_test) / total_inference_time_s

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
fnr      = (fn / (fn + tp)) * 100 if (fn + tp) > 0 else 0
fpr      = (fp / (fp + tn)) * 100 if (fp + tn) > 0 else 0
accuracy = accuracy_score(y_test, y_pred) * 100

# ── Superclass breakdown ─────────────────────────────────────────────────────
results_df = pd.DataFrame({'Actual': s_test.values, 'Pred': y_pred})
print(f"\n🔍 Detailed Breakdown for XGBoost:")
breakdown = []
for cat in results_df['Actual'].unique():
    subset  = results_df[results_df['Actual'] == cat]
    total   = len(subset)
    correct = (subset['Pred'] == 0).sum() if cat == 'Benign' else (subset['Pred'] == 1).sum()
    rate    = (correct / total) * 100 if total > 0 else 0
    breakdown.append({
        "Superclass"     : cat,
        "Samples"        : total,
        "Detection Rate" : f"{rate:.2f}%",
        "Missed"         : total - correct
    })
print(pd.DataFrame(breakdown).sort_values("Samples", ascending=False).to_string(index=False))

print(f"\n  ⏱️  Total Inference Time : {total_inference_time_s:.4f}s")
print(f"  ⚡  Throughput           : {throughput:,.0f} samples/sec")
print(f"  🎯  Accuracy             : {accuracy:.4f}%")

summary = {
    "Model"               : "XGBoost",
    "Accuracy (%)"        : round(accuracy, 4),
    "F1-Score"            : round(f1_score(y_test, y_pred), 4),
    "Recall (DR)"         : round(recall_score(y_test, y_pred), 4),
    "FNR"                 : f"{fnr:.3f}%",
    "FPR"                 : f"{fpr:.3f}%",
    "Latency (ms)"        : round(avg_latency_ms, 6),
    "Throughput (samp/s)" : round(throughput, 2),
    "Inference Time (s)"  : round(total_inference_time_s, 4),
}
print("\n" + "="*80)
print("XGBOOST PERFORMANCE SUMMARY")
print("="*80)
print(pd.DataFrame([summary]).to_string(index=False))

# ── Save .pkl to Drive ────────────────────────────────────────────────────────
save_dir  = '/content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smotenc(hybrid70-30)/models'
os.makedirs(save_dir, exist_ok=True)
pkl_path  = os.path.join(save_dir, 'xgboost_model.pkl')
joblib.dump(model, pkl_path)
size_kb   = os.path.getsize(pkl_path) / 1024
print(f"\n💾 Model saved to : {pkl_path}")
print(f"📦 .pkl size      : {size_kb:.2f} KB")

🚀 Training XGBoost...

🔍 Detailed Breakdown for XGBoost:
 Superclass  Samples Detection Rate  Missed
     Benign   197520         95.25%    9391
       DDoS   139841        100.00%       6
      Recon    59609         59.00%   24439
        DoS    46014        100.00%       0
      Mirai    40406        100.00%       1
  Web-based    34630         98.72%     442
   Spoofing    26920         79.01%    5650
Brute Force     5766         97.99%     116
        NaN        0          0.00%       0

  ⏱️  Total Inference Time : 0.6003s
  ⚡  Throughput           : 927,066 samples/sec
  🎯  Accuracy             : 92.8039%

XGBOOST PERFORMANCE SUMMARY
  Model  Accuracy (%)  F1-Score  Recall (DR)    FNR    FPR  Latency (ms)  Throughput (samp/s)  Inference Time (s)
XGBoost       92.8039    0.9425       0.9146 8.540% 4.754%      0.001079            927066.38              0.6003

💾 Model saved to : /content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smotenc(hybrid70-30)/models/xgboost_model.pkl


In [6]:
import os
import joblib
import bz2
import pickle
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from onnxmltools import convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType
import onnxruntime as rt

# ── Paths ────────────────────────────────────────────────────────────────────
save_dir    = '/content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smotenc(hybrid70-30)/models'
pkl_path    = os.path.join(save_dir, 'xgboost_model.pkl')
joblib_path = os.path.join(save_dir, 'xgboost_model_compressed.pkl')
bz2_path    = os.path.join(save_dir, 'xgboost_model_compressed.pbz2')
onnx_path   = os.path.join(save_dir, 'xgboost_model.onnx')

# ── Load original model ───────────────────────────────────────────────────────
model       = joblib.load(pkl_path)
original_kb = os.path.getsize(pkl_path) / 1024

# ── Conversion 1: joblib lz4 compressed ──────────────────────────────────────
joblib.dump(model, joblib_path, compress=('lz4', 9))
joblib_kb = os.path.getsize(joblib_path) / 1024

# ── Conversion 2: bz2 pickle ─────────────────────────────────────────────────
with bz2.BZ2File(bz2_path, 'wb') as f:
    pickle.dump(model, f)
bz2_kb = os.path.getsize(bz2_path) / 1024

# ── Conversion 3: ONNX — retrain with integer feature names ──────────────────
# onnxmltools requires feature names in f0/f1/f2 format
# Solution: retrain an identical model on the same data with renamed columns

print("🔄 Retraining XGBoost with integer feature names for ONNX conversion...")
n_features    = X_train.shape[1]
int_col_names = [f"f{i}" for i in range(n_features)]

X_train_int = X_train.copy()
X_test_int  = X_test.copy()
X_train_int.columns = int_col_names
X_test_int.columns  = int_col_names

model_for_onnx = XGBClassifier(n_estimators=50, max_depth=6, verbosity=0)
model_for_onnx.fit(X_train_int, y_train)
print("✅ Done. Converting to ONNX...")

initial_type = [('float_input', FloatTensorType([None, n_features]))]
onnx_model   = convert_xgboost(model_for_onnx, initial_types=initial_type)
with open(onnx_path, 'wb') as f:
    f.write(onnx_model.SerializeToString())
onnx_kb = os.path.getsize(onnx_path) / 1024

# ── Verify ONNX predictions match original ────────────────────────────────────
sess       = rt.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
input_name = sess.get_inputs()[0].name
label_name = sess.get_outputs()[0].name
y_onnx     = sess.run([label_name], {input_name: X_test_int.values.astype(np.float32)})[0]
match_pct  = (y_onnx == y_pred).mean() * 100

# ── Size comparison table ─────────────────────────────────────────────────────
rows = [
    {"Format"              : "Original .pkl (joblib)",
     "Size (KB)"           : round(original_kb, 2),
     "Reduction"           : "— baseline",
     "Predictions Match"   : "✅ baseline"},
    {"Format"              : "joblib lz4 compressed .pkl",
     "Size (KB)"           : round(joblib_kb, 2),
     "Reduction"           : f"{(1 - joblib_kb/original_kb)*100:.1f}% smaller",
     "Predictions Match"   : "✅ identical"},
    {"Format"              : "bz2 compressed .pbz2",
     "Size (KB)"           : round(bz2_kb, 2),
     "Reduction"           : f"{(1 - bz2_kb/original_kb)*100:.1f}% smaller",
     "Predictions Match"   : "✅ identical"},
    {"Format"              : "ONNX (.onnx)",
     "Size (KB)"           : round(onnx_kb, 2),
     "Reduction"           : f"{(1 - onnx_kb/original_kb)*100:.1f}% smaller",
     "Predictions Match"   : f"✅ {match_pct:.4f}% agreement"},
]

print("\n" + "="*90)
print("MODEL SIZE COMPARISON AFTER CONVERSION")
print("="*90)
print(pd.DataFrame(rows).to_string(index=False))
print("\n📁 All files saved to:", save_dir)

🔄 Retraining XGBoost with integer feature names for ONNX conversion...
✅ Done. Converting to ONNX...

MODEL SIZE COMPARISON AFTER CONVERSION
                    Format  Size (KB)     Reduction     Predictions Match
    Original .pkl (joblib)     189.78    — baseline            ✅ baseline
joblib lz4 compressed .pkl      63.65 66.5% smaller           ✅ identical
      bz2 compressed .pbz2      63.33 66.6% smaller           ✅ identical
              ONNX (.onnx)     140.38 26.0% smaller ✅ 100.0000% agreement

📁 All files saved to: /content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smotenc(hybrid70-30)/models


In [4]:
import numpy as np
import pandas as pd
import joblib
import bz2
import pickle
import onnxruntime as rt
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# ── Load all 3 variants ───────────────────────────────────────────────────────
model_original = joblib.load(pkl_path)

model_joblib   = joblib.load(joblib_path)

with bz2.BZ2File(bz2_path, 'rb') as f:
    model_bz2 = pickle.load(f)

sess       = rt.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
input_name = sess.get_inputs()[0].name
label_name = sess.get_outputs()[0].name

# ── Get predictions from all variants ────────────────────────────────────────
y_orig   = model_original.predict(X_test)
y_jlib   = model_joblib.predict(X_test)
y_bz2    = model_bz2.predict(X_test)
y_onnx   = sess.run([label_name], {input_name: X_test_int.values.astype(np.float32)})[0]

# ── CHECK 1: Prediction agreement (sample-level) ──────────────────────────────
print("=" * 60)
print("CHECK 1 — Prediction Agreement (every sample)")
print("=" * 60)
checks = {
    "joblib compressed vs original" : y_jlib,
    "bz2 compressed vs original"    : y_bz2,
    "ONNX vs original"              : y_onnx,
}
for label, y_var in checks.items():
    n_diff    = (y_var != y_orig).sum()
    agree_pct = (y_var == y_orig).mean() * 100
    print(f"  {label:<35} → {agree_pct:.6f}% match  |  {n_diff} differing predictions")

# ── CHECK 2: Key metrics are identical ───────────────────────────────────────
print("\n" + "=" * 60)
print("CHECK 2 — Performance Metrics")
print("=" * 60)

def get_metrics(y_true, y_pred, name):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "Model"      : name,
        "Accuracy"   : f"{accuracy_score(y_true, y_pred)*100:.6f}%",
        "F1"         : f"{f1_score(y_true, y_pred):.6f}",
        "FNR"        : f"{fn/(fn+tp)*100:.6f}%",
        "FPR"        : f"{fp/(fp+tn)*100:.6f}%",
    }

metric_rows = [
    get_metrics(y_test, y_orig, "Original .pkl"),
    get_metrics(y_test, y_jlib, "joblib compressed"),
    get_metrics(y_test, y_bz2,  "bz2 compressed"),
    get_metrics(y_test, y_onnx, "ONNX"),
]
print(pd.DataFrame(metric_rows).to_string(index=False))

# ── CHECK 3: Edge cases — probe with boundary samples ────────────────────────
print("\n" + "=" * 60)
print("CHECK 3 — Edge Case Samples")
print("=" * 60)

# Samples where original model was least confident (closest to 0.5 threshold)
proba_orig  = model_original.predict_proba(X_test)[:, 1]
edge_idx    = np.argsort(np.abs(proba_orig - 0.5))[:50]   # 50 most uncertain

edge_orig   = y_orig[edge_idx]
edge_jlib   = y_jlib[edge_idx]
edge_bz2    = y_bz2[edge_idx]
edge_onnx   = y_onnx[edge_idx]

print(f"  Testing on 50 most uncertain predictions (proba closest to 0.5):")
print(f"  joblib vs original : {(edge_jlib == edge_orig).sum()}/50 match")
print(f"  bz2    vs original : {(edge_bz2  == edge_orig).sum()}/50 match")
print(f"  ONNX   vs original : {(edge_onnx == edge_orig).sum()}/50 match")

# ── CHECK 4: Probability scores (not just class labels) ───────────────────────
print("\n" + "=" * 60)
print("CHECK 4 — Probability Score Drift (joblib & bz2 only)")
print("=" * 60)
# ONNX skipped here — it uses integer feature names so proba would need X_test_int
proba_jlib = model_joblib.predict_proba(X_test)[:, 1]
proba_bz2  = model_bz2.predict_proba(X_test)[:, 1]

print(f"  joblib vs original — max proba diff : {np.abs(proba_jlib - proba_orig).max():.10f}")
print(f"  bz2    vs original — max proba diff : {np.abs(proba_bz2  - proba_orig).max():.10f}")
print(f"  (0.0000000000 = byte-for-byte identical model)")

# ── VERDICT ───────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("VERDICT")
print("=" * 60)
all_match = all([
    (y_jlib == y_orig).all(),
    (y_bz2  == y_orig).all(),
    (y_onnx == y_orig).all(),
])
print(f"  All 3 compressed formats produce identical predictions: {'✅ YES' if all_match else '❌ NO — check diffs above'}")

CHECK 1 — Prediction Agreement (every sample)
  joblib compressed vs original       → 100.000000% match  |  0 differing predictions
  bz2 compressed vs original          → 100.000000% match  |  0 differing predictions
  ONNX vs original                    → 100.000000% match  |  0 differing predictions

CHECK 2 — Performance Metrics
            Model   Accuracy       F1       FNR       FPR
    Original .pkl 92.803874% 0.942518 8.539670% 4.754455%
joblib compressed 92.803874% 0.942518 8.539670% 4.754455%
   bz2 compressed 92.803874% 0.942518 8.539670% 4.754455%
             ONNX 92.803874% 0.942518 8.539670% 4.754455%

CHECK 3 — Edge Case Samples
  Testing on 50 most uncertain predictions (proba closest to 0.5):
  joblib vs original : 50/50 match
  bz2    vs original : 50/50 match
  ONNX   vs original : 50/50 match

CHECK 4 — Probability Score Drift (joblib & bz2 only)
  joblib vs original — max proba diff : 0.0000000000
  bz2    vs original — max proba diff : 0.0000000000
  (0.00000000